# Module 06 — Relational Databases: Module Assessment (Solution)

**Save Your Work**

Before you begin, save a copy of this notebook to your Google Drive:
**File > Save a copy in Drive**

---

This is the **solution notebook**. It contains complete, working SQL and Python code
for all tasks with expected outputs shown as comments.

## Setup — Imports and Connection

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON')
print('Connection established.')

---

## Task 1: Schema Creation

In [ ]:
# Task 1: CREATE TABLE statements

cursor.execute("""
CREATE TABLE IF NOT EXISTS genres (
    genre_id   INTEGER PRIMARY KEY,
    genre_name TEXT UNIQUE NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS movies (
    movie_id     INTEGER PRIMARY KEY,
    title        TEXT NOT NULL,
    genre_id     INTEGER REFERENCES genres(genre_id),
    release_year INTEGER,
    rental_price REAL CHECK (rental_price > 0)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    email       TEXT UNIQUE,
    join_date   TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS rentals (
    rental_id   INTEGER PRIMARY KEY,
    customer_id INTEGER REFERENCES customers(customer_id),
    movie_id    INTEGER REFERENCES movies(movie_id),
    rental_date TEXT NOT NULL,
    return_date TEXT
)
""")

conn.commit()
print('Tables created.')

---

## Task 2: Load Sample Data

In [ ]:
# Sample data — do not modify

genres_data = [
    (1, 'Action'),
    (2, 'Comedy'),
    (3, 'Drama'),
    (4, 'Sci-Fi'),
]

customers_data = [
    (1, 'Alice Martin',   'alice@example.com',   '2022-01-15'),
    (2, 'Bob Chen',       'bob@example.com',     '2022-03-20'),
    (3, 'Carol Davis',    'carol@example.com',   '2022-06-01'),
    (4, 'David Kim',      'david@example.com',   '2023-01-10'),
    (5, 'Eva Rossi',      'eva@example.com',     '2023-04-05'),
    (6, 'Frank Nguyen',   'frank@example.com',   '2023-07-22'),
    (7, 'Grace Patel',    'grace@example.com',   '2024-01-08'),
    (8, 'Henry Okonkwo',  'henry@example.com',   '2024-03-14'),
]

movies_data = [
    (1,  'The Speed Chase',      1, 2021, 3.99),
    (2,  'Laugh Factory',        2, 2019, 2.99),
    (3,  'Midnight Drama',       3, 2020, 3.49),
    (4,  'Galaxy Raiders',       4, 2022, 4.49),
    (5,  'Double Punch',         1, 2018, 2.99),
    (6,  'Office Chaos',         2, 2023, 3.99),
    (7,  'Silent Witness',       3, 2021, 3.49),
    (8,  'Quantum Break',        4, 2020, 4.99),
    (9,  'Iron Storm',           1, 2023, 4.49),
    (10, 'The Reunion',          3, 2019, 2.99),
    (11, 'Neon Frontier',        4, 2022, 4.99),
    (12, 'Comedy Night Live',    2, 2024, 3.49),
]

rentals_data = [
    (1,  1, 1,  '2024-01-05', '2024-01-08'),
    (2,  1, 4,  '2024-01-10', '2024-01-13'),
    (3,  2, 2,  '2024-01-12', '2024-01-14'),
    (4,  2, 6,  '2024-01-20', '2024-01-22'),
    (5,  3, 3,  '2024-02-01', '2024-02-04'),
    (6,  3, 8,  '2024-02-05', '2024-02-09'),
    (7,  3, 11, '2024-02-15', '2024-02-18'),
    (8,  4, 1,  '2024-02-20', '2024-02-23'),
    (9,  4, 9,  '2024-03-01', '2024-03-04'),
    (10, 5, 4,  '2024-03-05', '2024-03-08'),
    (11, 5, 12, '2024-03-10', '2024-03-12'),
    (12, 6, 5,  '2024-03-15', '2024-03-18'),
    (13, 6, 7,  '2024-03-20', '2024-03-23'),
    (14, 7, 2,  '2024-04-01', '2024-04-03'),
    (15, 7, 10, '2024-04-05', '2024-04-08'),
    (16, 1, 8,  '2024-04-10', '2024-04-14'),
    (17, 2, 11, '2024-04-15', '2024-04-18'),
    (18, 3, 6,  '2024-04-20', '2024-04-23'),
    (19, 5, 3,  '2024-05-01', '2024-05-04'),
    (20, 6, 9,  '2024-05-10', None),
]

In [ ]:
# Task 2: Insert data using executemany() with ? placeholders

cursor.executemany('INSERT INTO genres VALUES (?, ?)', genres_data)
cursor.executemany('INSERT INTO customers VALUES (?, ?, ?, ?)', customers_data)
cursor.executemany('INSERT INTO movies VALUES (?, ?, ?, ?, ?)', movies_data)
cursor.executemany('INSERT INTO rentals VALUES (?, ?, ?, ?, ?)', rentals_data)

conn.commit()
print('Data loaded.')
print('Genres:', cursor.execute('SELECT COUNT(*) FROM genres').fetchone()[0])   # 4
print('Movies:', cursor.execute('SELECT COUNT(*) FROM movies').fetchone()[0])   # 12
print('Customers:', cursor.execute('SELECT COUNT(*) FROM customers').fetchone()[0])  # 8
print('Rentals:', cursor.execute('SELECT COUNT(*) FROM rentals').fetchone()[0])  # 20

---

## Task 3: Basic Queries

In [ ]:
# Query 3a: All movies ordered by rental_price descending

sql_3a = """
SELECT title, release_year, rental_price
FROM movies
ORDER BY rental_price DESC
"""
pd.read_sql_query(sql_3a, conn)
# Expected: 12 rows, Quantum Break and Neon Frontier at top (4.99)

In [ ]:
# Query 3b: Customers who joined in 2023 or later

sql_3b = """
SELECT name, join_date
FROM customers
WHERE join_date >= '2023-01-01'
ORDER BY join_date ASC
"""
pd.read_sql_query(sql_3b, conn)
# Expected: David Kim, Eva Rossi, Frank Nguyen, Grace Patel, Henry Okonkwo

In [ ]:
# Query 3c: Top 5 most expensive movies

sql_3c = """
SELECT title, rental_price
FROM movies
ORDER BY rental_price DESC
LIMIT 5
"""
pd.read_sql_query(sql_3c, conn)

In [ ]:
# Query 3d: Rentals not yet returned

sql_3d = """
SELECT rental_id, customer_id, movie_id, rental_date
FROM rentals
WHERE return_date IS NULL
"""
pd.read_sql_query(sql_3d, conn)
# Expected: 1 row — rental_id 20, Frank Nguyen, Iron Storm

In [ ]:
# Query 3e: Movie count per genre

sql_3e = """
SELECT genre_id, COUNT(*) AS movie_count
FROM movies
GROUP BY genre_id
ORDER BY movie_count DESC
"""
pd.read_sql_query(sql_3e, conn)
# Expected: 4 rows — each genre has 3 movies

---

## Task 4: Advanced Queries

In [ ]:
# Query 4a: CASE WHEN — price tier

sql_4a = """
SELECT
    title,
    rental_price,
    CASE
        WHEN rental_price < 3.00             THEN 'Budget'
        WHEN rental_price BETWEEN 3.00 AND 3.99 THEN 'Standard'
        ELSE 'Premium'
    END AS price_tier
FROM movies
ORDER BY rental_price
"""
pd.read_sql_query(sql_4a, conn)

In [ ]:
# Query 4b: COALESCE — return status

sql_4b = """
SELECT
    rental_id,
    movie_id,
    rental_date,
    COALESCE(return_date, 'Not returned') AS return_status
FROM rentals
ORDER BY rental_id
"""
pd.read_sql_query(sql_4b, conn)
# Expected: rental_id 20 shows 'Not returned'

In [ ]:
# Query 4c: Subquery — movies above average rental price

sql_4c = """
SELECT title, rental_price
FROM movies
WHERE rental_price > (SELECT AVG(rental_price) FROM movies)
ORDER BY rental_price DESC
"""
pd.read_sql_query(sql_4c, conn)
# Average is ~3.82; movies >= 3.99 are above average

---

## Task 5: Aggregation

In [ ]:
# Query 5a: Rentals and revenue per genre

sql_5a = """
SELECT
    g.genre_name,
    COUNT(r.rental_id)      AS total_rentals,
    ROUND(SUM(m.rental_price), 2) AS total_revenue
FROM rentals r
JOIN movies   m ON r.movie_id  = m.movie_id
JOIN genres   g ON m.genre_id  = g.genre_id
GROUP BY g.genre_name
ORDER BY total_revenue DESC
"""
pd.read_sql_query(sql_5a, conn)

In [ ]:
# Query 5b: Top spending customer

sql_5b = """
SELECT
    c.name,
    ROUND(SUM(m.rental_price), 2) AS total_spent
FROM rentals  r
JOIN customers c ON r.customer_id = c.customer_id
JOIN movies    m ON r.movie_id    = m.movie_id
GROUP BY c.customer_id, c.name
ORDER BY total_spent DESC
LIMIT 1
"""
pd.read_sql_query(sql_5b, conn)

In [ ]:
# Query 5c: Monthly rental counts with HAVING

sql_5c = """
SELECT
    strftime('%Y-%m', rental_date) AS year_month,
    COUNT(*) AS rental_count
FROM rentals
GROUP BY year_month
HAVING rental_count > 2
ORDER BY year_month
"""
pd.read_sql_query(sql_5c, conn)
# Expected: months with > 2 rentals (Jan 4, Feb 4, Mar 4, Apr 4, May 2 — May excluded)

---

## Task 6: Joins

In [ ]:
# Query 6a: INNER JOIN — full rental details

sql_6a = """
SELECT
    c.name       AS customer_name,
    m.title      AS movie_title,
    r.rental_date,
    r.return_date,
    m.rental_price
FROM rentals   r
JOIN customers c ON r.customer_id = c.customer_id
JOIN movies    m ON r.movie_id    = m.movie_id
ORDER BY r.rental_date ASC
"""
pd.read_sql_query(sql_6a, conn)

In [ ]:
# Query 6b: LEFT JOIN — customers with no rentals

sql_6b = """
SELECT
    c.name,
    r.rental_id
FROM customers c
LEFT JOIN rentals r ON c.customer_id = r.customer_id
WHERE r.rental_id IS NULL
"""
pd.read_sql_query(sql_6b, conn)
# Expected: Henry Okonkwo (customer_id 8) has no rentals

In [ ]:
# Query 6c: Three-table join — spending per customer per genre

sql_6c = """
SELECT
    c.name                        AS customer_name,
    g.genre_name,
    ROUND(SUM(m.rental_price), 2) AS total_spent
FROM rentals   r
JOIN customers c ON r.customer_id = c.customer_id
JOIN movies    m ON r.movie_id    = m.movie_id
JOIN genres    g ON m.genre_id    = g.genre_id
GROUP BY c.customer_id, c.name, g.genre_id, g.genre_name
ORDER BY c.name ASC, total_spent DESC
"""
pd.read_sql_query(sql_6c, conn)

---

## Task 7: Python + Pandas Integration

In [ ]:
# Task 7a: Load genre revenue results into Pandas

sql_genre_revenue = """
SELECT
    g.genre_name,
    COUNT(r.rental_id)            AS total_rentals,
    ROUND(SUM(m.rental_price), 2) AS total_revenue
FROM rentals r
JOIN movies  m ON r.movie_id  = m.movie_id
JOIN genres  g ON m.genre_id  = g.genre_id
GROUP BY g.genre_name
ORDER BY total_revenue DESC
"""

df_genre = pd.read_sql_query(sql_genre_revenue, conn)
print(df_genre.describe())
df_genre

In [ ]:
# Task 7b: Bar chart — total revenue by genre

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(df_genre['genre_name'], df_genre['total_revenue'], color='steelblue', edgecolor='white')
ax.set_title('Total Rental Revenue by Genre')
ax.set_xlabel('Genre')
ax.set_ylabel('Total Revenue (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# Task 7c: Export to CSV

df_genre.to_csv('genre_revenue.csv', index=False)
print('Saved genre_revenue.csv')
print(pd.read_csv('genre_revenue.csv'))